# Lab 1: Naive Bayes Classifier — Predicting Whether to Play Tennis

## 1. Objective
Build a simple **Naive Bayes classifier** to predict whether a person should play tennis (`Yes`/`No`) based on weather conditions: **Outlook**, **Temperature**, **Humidity**, and **Wind**.

## 2. Training Dataset

| Day | Outlook  | Temperature | Humidity | Wind   | PlayTennis |
|-----|----------|-------------|----------|--------|------------|
| D1  | Sunny    | Hot         | High     | Weak   | No         |
| D2  | Sunny    | Hot         | High     | Strong | No         |
| D3  | Overcast | Hot         | High     | Weak   | Yes        |
| D4  | Rain     | Mild        | High     | Weak   | Yes        |
| D5  | Rain     | Cool        | Normal   | Weak   | Yes        |
| D6  | Rain     | Cool        | Normal   | Strong | No         |
| D7  | Overcast | Cool        | Normal   | Strong | Yes        |
| D8  | Overcast | Mild        | High     | Weak   | No         |
| D9  | Sunny    | Cool        | Normal   | Weak   | Yes        |
| D10 | Rain     | Mild        | Normal   | Weak   | Yes        |

## 3. Test Dataset

| No. | Outlook  | Temperature | Humidity | Wind   | Class |
|-----|----------|-------------|----------|--------|-------|
| X1  | Sunny    | Mild        | Normal   | Strong | ???   |
| X2  | Rain     | Hot         | High     | Strong | ???   |
| X3  | Overcast | Cool        | High     | Strong | ???   |
| X4  | Rain     | Mild        | Normal   | Strong | ???   |
| X5  | Sunny    | Cool        | High     | Weak   | ???   |
| X6  | Overcast | Mild        | Normal   | Weak   | ???   |
| X7  | Sunny    | Hot         | Normal   | Weak   | ???   |
| X8  | Rain     | Cool        | High     | Weak   | ???   |
| X9  | Overcast | Hot         | Normal   | Strong | ???   |
| X10 | Sunny    | Mild        | High     | Weak   | ???   |

## 4. Theoretical Background

By **Bayes' Theorem**:

$$
P(c \mid X) = \frac{P(X \mid c) \cdot P(c)}{P(X)}
$$

where:
- $c$ — class label ($c \in \{\text{Yes}, \text{No}\}$)
- $X = \{x_1, x_2, \dots, x_n\}$ — feature vector (Outlook, Temperature, Humidity, Wind)

Since $P(X)$ is constant for all classes, comparing classes only requires the numerator:

$$
P(c \mid X) \propto P(X \mid c) \cdot P(c)
$$

Applying the **Naive Bayes assumption** — that features are conditionally independent given the class — we get:

$$
P(c \mid X) \propto P(c) \cdot \prod_{i=1}^{n} P(x_i \mid c)
$$

$$
P(c \mid X) \propto P(c) \cdot P(x_1 \mid c) \cdot P(x_2 \mid c) \cdots P(x_n \mid c)
$$

## 5. Classification Steps
1. Compute the prior probabilities $P(\text{Yes})$ and $P(\text{No})$ from the training set.
2. Compute the conditional probability of each feature value given each class, e.g. $P(\text{Outlook} = \text{Sunny} \mid \text{Yes})$.
3. For each test sample, compute the (unnormalized) posterior score for both classes using the product formula above.
4. Assign the class with the **higher** score.

In [8]:
pip install numpy


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import math

import numpy as np  

In [10]:
def create_training_data():
  data = [["Sunny", "Hot", "High", "Weak", "No"],
          ["Sunny", "Hot", "High", "Strong", "No"],
          ["Overcast", "Hot", "High", "Weak", "Yes"],
          ["Rain", "Mild", "High", "Weak", "Yes"],
          ["Rain", "Cool", "Normal", "Weak", "Yes"],
          ["Rain", "Cool", "Normal", "Strong", "No"],
          ["Overcast", "Cool", "Normal", "Strong", "Yes"],
          ["Overcast", "Mild", "High", "Weak", "No"],
          ["Sunny", "Cool", "Normal", "Weak", "Yes"],
          ["Rain", "Mild", "Normal", "Weak", "Yes"]]
  return np.array(data)

train_data = create_training_data()
print(train_data.shape)
print(train_data)

(10, 5)
[['Sunny' 'Hot' 'High' 'Weak' 'No']
 ['Sunny' 'Hot' 'High' 'Strong' 'No']
 ['Overcast' 'Hot' 'High' 'Weak' 'Yes']
 ['Rain' 'Mild' 'High' 'Weak' 'Yes']
 ['Rain' 'Cool' 'Normal' 'Weak' 'Yes']
 ['Rain' 'Cool' 'Normal' 'Strong' 'No']
 ['Overcast' 'Cool' 'Normal' 'Strong' 'Yes']
 ['Overcast' 'Mild' 'High' 'Weak' 'No']
 ['Sunny' 'Cool' 'Normal' 'Weak' 'Yes']
 ['Rain' 'Mild' 'Normal' 'Weak' 'Yes']]


In [11]:
def compute_prior_probabilities(train_data):
  class_names = ["No", "Yes"]
  total_samples = len(train_data)
  prior_probs = np.zeros(len(class_names)) #2

  for id_val, class_name in enumerate(class_names):
    for i in range(total_samples):
      if (train_data[i][4] == class_name):
        prior_probs[id_val] += 1
    prior_probs[id_val] /= total_samples
  return prior_probs

prior_probability = compute_prior_probabilities(train_data)
print("P(‘Play Tennis’ = No)", prior_probability[0])
print("P(‘Play Tennis’ = Yes)", prior_probability[1])

P(‘Play Tennis’ = No) 0.4
P(‘Play Tennis’ = Yes) 0.6


In [13]:
from enum import unique
def compute_conditional_probabilities(train_data):
  # P(feat | class)
  class_names = ["No", "Yes"]
  n_features = train_data.shape[1] - 1 # 5 - 1
  conditional_probs = []
  feature_values = []

  for f_id in range(n_features):
    unique_val = np.unique(train_data[:, f_id])
    all_val = np.array(train_data[:, [f_id, -1]])
    #unique_val = [Sunny, Overcast, Rain]
    #print(all_val)
    #print(unique_val)
    feature_values.append(unique_val)

    feat_cond_probs = np.zeros((len(class_names), len(unique_val)))
    #print(feat_cond_probs)
    # 0 0 0
    # 0 0 0


    for class_id, class_name in enumerate(class_names):
      # enumerate: gan them idx cho class_names
      #print("->", end="")
      #print(class_id, class_name)
      # code
      # get samples for this class?

      # dem tan xuat cua x
      for val_id, vals in enumerate(unique_val):
        cnt_val = 0
        for i in range(len(all_val)):
          if (all_val[i][0] == vals):
            if (all_val[i][1] == class_name):
              cnt_val += 1
          feat_cond_probs[class_id][val_id] = (cnt_val / (prior_probability[class_id] * 10))

    #print(feat_cond_probs)

    conditional_probs.append(feat_cond_probs)
    #print(conditional_probs)

  return conditional_probs, feature_values

_, feature_values = compute_conditional_probabilities(train_data)
print("x1 = ",feature_values[0])
print("x2 = ",feature_values[1])
print("x3 = ",feature_values[2])
print("x4 = ",feature_values[3])


x1 =  ['Overcast' 'Rain' 'Sunny']
x2 =  ['Cool' 'Hot' 'Mild']
x3 =  ['High' 'Normal']
x4 =  ['Strong' 'Weak']


In [14]:
def get_feature_index(feature_value, feature_values):
  for idx, val in enumerate(feature_values):
    if (val == feature_value):
      return idx


_, feature_values = compute_conditional_probabilities(train_data)
outlook = feature_values[1]

i1 = get_feature_index("Hot", outlook)
i2 = get_feature_index("Cool", outlook)
i3 = get_feature_index("Mild", outlook)

print(i1, i2, i3)


1 0 2


In [15]:
def train_naive_bayes(train_data):
  prior_probabilities = compute_prior_probabilities(train_data)

  conditional_probabilities, feature_names = compute_conditional_probabilities(train_data)

  return prior_probabilities, conditional_probabilities, feature_names

prior_probs, conditional_probs, feature_names =  train_naive_bayes(
train_data)

print(prior_probs)
print(conditional_probs)
print(feature_names)

[0.4 0.6]
[array([[0.25      , 0.25      , 0.5       ],
       [0.33333333, 0.5       , 0.16666667]]), array([[0.25      , 0.5       , 0.25      ],
       [0.5       , 0.16666667, 0.33333333]]), array([[0.75      , 0.25      ],
       [0.33333333, 0.66666667]]), array([[0.5       , 0.5       ],
       [0.16666667, 0.83333333]])]
[array(['Overcast', 'Rain', 'Sunny'], dtype='<U8'), array(['Cool', 'Hot', 'Mild'], dtype='<U8'), array(['High', 'Normal'], dtype='<U8'), array(['Strong', 'Weak'], dtype='<U8')]


In [16]:
def predict_tennis(X, prior_probabilities, conditional_probabilities, feature_names):
  class_names = ["No", "Yes"]
  feature_indices = []
  for i, feature_value in enumerate(X):
    feature_indices.append(get_feature_index(feature_value, feature_names[i]))

  #print(prior_probabilities)

  #print(conditional_probabilities[Xi][class_name][Feature])

  class_probabilities = np.zeros(len(class_names))
  for class_idx in range(len(class_names)):
    class_probabilities[class_idx] = (prior_probabilities[class_idx])
    for idx, feat_idx in enumerate(feature_indices):
      class_probabilities[class_idx] *= conditional_probabilities[idx][class_idx][feat_idx]

  #print(class_probabilities)

  # normalize probabilities
  total_probs = sum(class_probabilities)
  if (total_probs > 0):
    normalize_probs = [p / total_probs for p in class_probabilities]
  else:
    normalize_probs = [0.5, 0.5]

  # predict
  predicted_class_idx = np.argmax(class_probabilities)
  #print(predicted_class_idx)
  prediction = class_names[predicted_class_idx]
  #print(prediction)
  #print(normalize_probs)

  prob_dict = {
      "No": round(normalize_probs[0].item(), 2),
      "Yes": round(normalize_probs[1].item(), 2)
  }

  return prediction, prob_dict


def input(X):
  prior_probs, conditional_probs, feature_names = train_naive_bayes(
  train_data)

  prediction, prob_dict = predict_tennis(X, prior_probs, conditional_probs, feature_names)
  if (prediction == "Yes"):
    return "Ad should go!"
  else:
    return "Ad should not go!"


X1 = ["Sunny", "Mild", "Normal", "Strong"]
print(input(X1))

X2 = ["Rain", "Hot", "High", "Strong"]
print(input(X2))

X3 = ["Overcast", "Cool", "High", "Strong"]
print(input(X3))

X4 = ["Rain", "Mild", "Normal", "Strong"]
print(input(X4))

X5 = ["Sunny", "Cool", "High", "Weak"]
print(input(X5))

X6 = ["Overcast", "Mild", "Normal", "Weak"]
print(input(X6))

X7 = ["Sunny", "Hot", "Normal", "Weak"]
print(input(X7))

X8 = ["Rain", "Cool", "High", "Weak"]
print(input(X8))

X9 = ["Overcast", "Hot", "Normal", "Strong"]
print(input(X9))

X10 = ["Sunny", "Mild", "High", "Weak"]
print(input(X10))

Ad should not go!
Ad should not go!
Ad should not go!
Ad should go!
Ad should not go!
Ad should go!
Ad should not go!
Ad should go!
Ad should not go!
Ad should not go!
